# Tutorial 06: a ligand in a protein pocket

*Author: Rhyan Barrett*

p-xylene in T4 lysozyme L99A in explicit water. The ligand is FieldMACE (DFT quality) in the field of the protein,
water and ions; the rest is Amber14/TIP3P. The labelled data are shipped: steps 2-3 are optional. Details:
[README](README.md).

In [ ]:
import os, subprocess, sys
import numpy as np
from IPython.display import Image, display

QUICK = os.environ.get("FIELDMACE_TUTORIAL_QUICK") == "1"   # tiny settings for automatic checks
os.environ.setdefault("OMP_NUM_THREADS", "4")

def run(cmd, env=None):
    """Run a tutorial step and show its output (stops the notebook if the step fails)."""
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       env={**os.environ, **(env or {})})
    lines = [l for l in p.stdout.splitlines() if "Warning" not in l and "warnings.warn" not in l]
    print("\n".join(lines[-60:]))
    if p.returncode:
        raise RuntimeError(f"step failed: {cmd}")

## Step 1: the solvated complex, with GAFF parameters for the ligand

In [ ]:
run(f"python 01_prepare_system.py --equil-ps {0.2 if QUICK else 50}")

In [ ]:
print(open("work/setup.json").read()[:600])

## Steps 2-3 (optional): sampling and PySCF labels

A few frames here as a demonstration (the full data set is shipped in `data/`).

In [ ]:
run(f"python 02_sample_configurations.py --n-frames {4 if QUICK else 20} --stride-ps {0.02 if QUICK else 0.5}")
run("python 03_label_with_pyscf.py --max-frames 2 --gas-fraction 0 --out work/demo_data --partial work/demo_partial.xyz"
    + (" --basis sto-3g" if QUICK else ""))

## Step 4: training

In [ ]:
env = {"EPOCHS": "2", "CHANNELS": "8"} if QUICK else {}
run("bash 04_train.sh", env=env)

## Step 5: evaluation, and what the pocket's field does

In [ ]:
run("python 05_evaluate.py")
display(Image("results/parity_pocket.png"))

## Step 6: ML/MM MD in OpenMM, compared with the classical ligand

In [ ]:
run(f"python 06_mlmm_md.py --nvt-ps {0.01 if QUICK else 2} --nve-ps {0.01 if QUICK else 0.5}")
display(Image("results/ligand_geometry.png"))